# 03 - Semantic and hybrid search

Build a BM25 + dense-vector index with `newsroom.search`, compare retrieval modes with the metrics in `newsroom.evaluation.search_eval`, and cluster the embedding space.

The default embedding provider is TF-IDF + TruncatedSVD (offline). Swap in `sentence_transformers` by installing the `transformers` extra and passing `provider_name="sentence_transformers"`.

In [ ]:
from newsroom.data.corpus import generate_corpus
from newsroom.search.query import build_search_engine

df = generate_corpus(n_docs=2000, seed=42)
engine = build_search_engine(df, provider_name="tfidf_svd", dimension=256)
len(engine)

## Querying in three modes

`bm25` is purely lexical, `vector` is purely semantic, and `hybrid` fuses both rankings with reciprocal rank fusion.

In [ ]:
query = "central bank raises interest rates as inflation persists"
for mode in ("bm25", "vector", "hybrid"):
    print(f"--- {mode}")
    for r in engine.search(query, top_k=3, mode=mode):
        print(f"  {r.rank}. [{r.label}] {r.title}  score={r.score:.4f}  {r.component_scores}")

## Evaluating retrieval

`build_relevance_set` samples documents, turns their TextRank keywords into queries, and marks same-label documents as relevant (the source document gets a higher gain). This gives Precision@k, Recall@k, MRR and nDCG@k without manual labelling.

In [ ]:
import pandas as pd

from newsroom.evaluation.search_eval import build_relevance_set, evaluate_search_engine

relevance = build_relevance_set(df, n_queries=60, seed=7)
results = evaluate_search_engine(engine, relevance, k=10)
pd.DataFrame({mode: m.to_dict() for mode, m in results.items()}).T.round(3)

## Incremental indexing and persistence

In [ ]:
new_docs = pd.DataFrame(
    {
        "doc_id": ["extra-1", "extra-2"],
        "title": ["Volcano erupts", "Lava cools"],
        "text": [
            "A volcanic eruption sent lava toward coastal villages as scientists monitored tremors.",
            "Lava flows slowed overnight while the observatory kept its sensors running.",
        ],
        "label": ["science", "science"],
    }
)
engine.add_documents(new_docs)
print([r.doc_id for r in engine.search("volcanic eruption lava", top_k=3)])
engine.save("../artifacts/index")

## Topic clustering on the embedding space

K-means with silhouette-based selection of `k`, plus the most distinctive terms per cluster.

In [ ]:
from newsroom.topics.clustering import TopicClusterer

texts = df["text"].tolist()
embeddings = engine.provider.embed_texts(texts)
clusters = TopicClusterer(k_range=(4, 6, 8)).fit(embeddings, texts)
print("k =", clusters.k, "silhouette =", round(clusters.silhouette, 3))
pd.DataFrame(clusters.summary())

In [ ]:
from IPython.display import Image

from newsroom.visualization.plots import plot_embedding_projection

Image(filename=str(plot_embedding_projection(embeddings, df["label"], "../artifacts/notebooks/projection.png", method="pca")))

## LDA topics

A bag-of-words topic model as a complementary, interpretable view.

In [ ]:
from newsroom.topics.lda import LdaTopicModel

lda = LdaTopicModel(n_topics=6, max_iter=10).fit(texts)
for topic in lda.summary(n_terms=8):
    print(topic["topic"], topic["terms"])